In [ ]:
import pandas as pd

movies  = pd.read_csv("data/raw/movies.csv")
ratings = pd.read_csv("data/raw/ratings.csv")
tags    = pd.read_csv("data/raw/tags.csv")
links   = pd.read_csv("data/raw/links.csv")

for nome, df in [("movies", movies), ("ratings", ratings), ("tags", tags), ("links", links)]:
    print(f"\n===== {nome} =====")
    print("Linhas, colunas:", df.shape)
    print("Colunas:", list(df.columns))
    print("Nulos por coluna:\n", df.isnull().sum())
    print(df.head(3))

Como os ratings tem 100836 entradas, e o enunciado pede apenas 100000, vamos tirar 836 das entradas de forma aleatória mas que seja reproduzivel no futuro.

In [ ]:
ratings = ratings.sample(n=100000, random_state=42).reset_index(drop=True)

print("Ratings depois de remover:", ratings.shape)
print(ratings.head(3))

Antes de limpar os dados, verificamos se há valores em falta no `ratings`, `tags` e `links` (o `movies` já foi confirmado sem nulos no output inicial).

In [ ]:
print(ratings.isnull().sum())
print(tags.isnull().sum())
print(links.isnull().sum())

Os 8 filmes sem `tmdbId` são mantidos, com o valor vazio (NULL), para não perdermos nem os filmes nem as suas notas. Convertemos a coluna para inteiro, porque os valores em falta tinham transformado em decimal.

In [ ]:
links["tmdbId"] = links["tmdbId"].astype("Int64")
print(links.head(3))

Verificamos se há duplicados: o `movieId` vai ser a chave primária dos filmes, por isso não pode repetir; e cada utilizador só deve ter uma nota por filme.

In [ ]:
print("movieId repetidos:", movies["movieId"].duplicated().sum())
print("Notas repetidas (mesmo user e filme):", ratings.duplicated(subset=["userId", "movieId"]).sum())

Alguns filmes não têm género: em vez de um valor vazio, têm o texto "(no genres listed)". Precisamos de ver quantos são.

In [ ]:
sem_genero = movies[movies["genres"] == "(no genres listed)"]
print("Filmes sem género:", len(sem_genero))
print(sem_genero.head())

Existem 34 filmes com "(no genres listed)". Mantemos estes filmes, mas sem nenhum género associado, para não criar um género falso. Isto vai ser tratado ao separar os géneros.

O ano do filme vem dentro do título, por exemplo "Toy Story (1995)". Antes de o separar numa coluna própria, verificamos se há títulos sem ano no fim.

In [ ]:
sem_ano = movies[~movies["title"].str.strip().str.contains(r"\(\d{4}\)$")]
print("Títulos sem ano:", len(sem_ano))
print(sem_ano)

A decisão ficou separar o ano do título numa coluna `year`. Os 13 títulos sem ano ficam com o ano vazio (NULL).

In [ ]:
movies["title"] = movies["title"].str.strip()
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")[0].astype("Int64")
movies["title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)

print(movies[["movieId", "title", "year"]].head(3))
print("Filmes sem ano:", movies["year"].isnull().sum())

O `timestamp` está em segundos desde 1970. Convertemo-lo para data e hora, no `ratings` e no `tags`, para ser legível e permitir queries por data

In [ ]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s")

print(ratings.head(3))
print(tags.head(3))

Transformamos o texto dos géneros (separados por "|") numa lista. Os filmes com "(no genres listed)" ficam com a lista vazia.

In [ ]:
movies["genres"] = movies["genres"].replace("(no genres listed)", "")
movies["genres"] = movies["genres"].str.split("|").apply(lambda g: [x for x in g if x != ""])

print(movies[["title", "genres"]].head(3))
print("Filmes com lista vazia:", (movies["genres"].str.len() == 0).sum())

Para o MySQL, a relação filme-género é many-to-many. Como já sabemos MySQL nao suporta lista dentro de colunas, entao o que fazemos é a partir da lista de géneros, criamos duas tabelas novas: `genres` (um id por género) e `movie_genres` (que liga cada filme aos seus géneros). O `movies` com a lista mantém-se para o MongoDB.

In [ ]:
movie_genres = movies[["movieId", "genres"]].explode("genres").dropna()
movie_genres = movie_genres.rename(columns={"genres": "genre"})

genres = pd.DataFrame({"genre": sorted(movie_genres["genre"].unique())})
genres["genreId"] = range(1, len(genres) + 1)

movie_genres = movie_genres.merge(genres, on="genre")[["movieId", "genreId"]]

print(genres)
print(movie_genres.head())
print("Ligações filme-género:", len(movie_genres))

Como cada filme tem apenas um `imdbId` e um `tmdbId` (one-to.one), juntamos o `links` ao `movies` em vez de criar uma tabela à parte.

In [ ]:
movies = movies.merge(links, on="movieId", how="left")
print(movies.head(3))

O dataset não tem ficheiro de users. Criamos a tabela `users` com todos os `userId` que aparecem no `ratings` e no `tags`, para que todas as notas e tags apontem para um utilizador que ja existe.

In [ ]:
user_ids = pd.concat([ratings["userId"], tags["userId"]]).unique()
users = pd.DataFrame({"userId": sorted(user_ids)})

print(users.head())
print("Utilizadores:", len(users))

Agora antes de gravar temos de verificar se tudo está a apontar corretamente: todas as notas e tags têm de apontar para filmes e utilizadores que existem.

In [ ]:
print("Ratings com filme inexistente:", (~ratings["movieId"].isin(movies["movieId"])).sum())
print("Tags com filme inexistente:", (~tags["movieId"].isin(movies["movieId"])).sum())
print("Ratings com user inexistente:", (~ratings["userId"].isin(users["userId"])).sum())
print("Tags com user inexistente:", (~tags["userId"].isin(users["userId"])).sum())

Visualizar como fica cada tabela

In [ ]:
for nome, df in [("movies", movies), ("genres", genres), ("movie_genres", movie_genres),
                 ("users", users), ("ratings", ratings), ("tags", tags)]:
    print(f"\n===== {nome} ===== {df.shape}")
    print(df.head(3))

Guardamos os dados limpos em `data/processed/`. Para o MySQL, cada tabela num CSV (o `movies` sem a lista de géneros). Para o MongoDB, o `movies` com a lista de géneros num JSON, porque o CSV não guarda listas.

In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)

# MySQL
movies.drop(columns=["genres"]).to_csv("data/processed/movies.csv", index=False)
genres.to_csv("data/processed/genres.csv", index=False)
movie_genres.to_csv("data/processed/movie_genres.csv", index=False)
users.to_csv("data/processed/users.csv", index=False)
ratings.to_csv("data/processed/ratings.csv", index=False)
tags.to_csv("data/processed/tags.csv", index=False)

# MongoDB
movies.to_json("data/processed/movies_mongo.json", orient="records", indent=2)

print(os.listdir("data/processed"))

adcionar uma csv de localizacao(nem que inventar)

na primeira etapa é melhor fazer uma mango tiver o mesmo numero de tabelas com sql (depois na quarta etapa melhrar pk depois é prcisa de melhorar isso.)

chegar ao goals 3

exprimentar com index, testar dados com ou sem index para ver se melhorou 

